# 証券営業インテリジェンス ハンズオン
## Part 3: Cortex Analyst（自然言語 to SQL）

このノートブックでは、**Cortex Analyst** のセットアップと動作確認を行います。
Cortex Analyst を使うと、SQLを書かなくても **自然言語で顧客データを検索・分析** できます。

### このパートで体験できること

1. **Semantic View の作成**: テーブルの意味・関係を定義
2. **Cortex Analyst のテスト**: 自然言語でSQLを自動生成
3. **Snowsight UI でのデモ**: 実際の営業担当者が使う画面を体験

### 🚀 体験ポイント

> **「SQLを書かずに、日本語で顧客データを分析できる！」**
>
> 「預かり資産5億円以上の顧客をリストアップして」と入力するだけで、
> 複数テーブルを結合した複雑なSQLが自動生成・実行されます。

### 前提条件
- `setup.sql` 実行済み（テーブル・ビュー作成・100名の顧客データ投入済み）

> ⏱️ **このパートの目安時間: 45分**

In [ ]:
%%sql -r result_env
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA, CURRENT_WAREHOUSE() AS WH;

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

images_dir = 'images/'

def display_image(image_file: str) -> None:
    image_path = os.path.join(images_dir, image_file)
    img = Image.open(image_path)
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 1. データ探索 - Semantic View の設計

Semantic View を作成する前に、**どのテーブルがあり、どのような関係にあるか** を確認します。
Semantic View はテーブルの意味・関係を定義するものなので、まずデータを理解することが重要です。

> ⏱️ **目安: 5分**

In [ ]:
%%sql -r result_tables
-- テーブル一覧とレコード数を確認
SELECT
    TABLE_NAME AS "テーブル名",
    ROW_COUNT AS "レコード数",
    COMMENT AS "テーブルの説明"
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'DEMO_SCHEMA'
  AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME;

In [ ]:
%%sql -r result_customer_sample
-- 顧客マスタのサンプル（山田様を含む上位5件）
SELECT
    CUSTOMER_ID,
    CUSTOMER_NAME,
    AGE,
    SEGMENT,
    TO_CHAR(TOTAL_ASSETS, '999,999,999,999') AS "総資産円",
    RISK_TOLERANCE AS "リスク許容度",
    INVESTMENT_PURPOSE AS "投資目的",
    LAST_CONTACT_DATE AS "最終接触日"
FROM DIM_CUSTOMER
ORDER BY TOTAL_ASSETS DESC
LIMIT 5;

In [ ]:
%%sql -r result_portfolio_sample
-- ポートフォリオのサンプル（山田様の保有銘柄）
SELECT
    p.CUSTOMER_ID,
    p.SECURITY_NAME AS "銘柄",
    p.ASSET_CLASS AS "資産クラス",
    TO_CHAR(p.MARKET_VALUE, '999,999,999') AS "時価評価額",
    p.UNREALIZED_GAIN AS "含み損益",
    ROUND(p.UNREALIZED_GAIN_PCT, 1) AS "含み損益率"
FROM FACT_PORTFOLIO p
WHERE p.CUSTOMER_ID = 'C001'
ORDER BY p.MARKET_VALUE DESC;

In [ ]:
-- 相続税試算ビューのサンプル（上位5名）
SELECT
    CUSTOMER_ID,
    CUSTOMER_NAME,
    AGE,
    TO_CHAR(ESTIMATED_TAX, '999,999,999,999') AS "推定相続税円",
    TO_CHAR(TOTAL_ASSETS, '999,999,999,999') AS "総資産円"
FROM V_INHERITANCE_TAX_ESTIMATE
ORDER BY ESTIMATED_TAX DESC
LIMIT 5;

## 2. Semantic View とは

**Semantic View** は、Cortex Analyst が「テーブルの意味」を理解するための **ビジネスメタデータ定義** です。

### なぜ必要か？

通常のデータベースには「列名」しかありません。
- `TOTAL_ASSETS` → Cortex Analyst: 「これは何？」
- Semantic View で `COMMENT = '総資産（円）'` と定義 → 「これが総資産だ！」と理解

### Semantic View の構成要素

| 要素 | 役割 | 例 |
|---|---|---|
| `TABLES` | 対象テーブルとその説明・プライマリキー | `CUSTOMERS AS DIM_CUSTOMER` |
| `RELATIONSHIPS` | テーブル間の関係（外部キー） | `CUSTOMERS → PORTFOLIO (CUSTOMER_ID)` |
| `SYNONYMS` | テーブル・列の別名（日本語対応） | `CUSTOMERS` の同義語に `顧客、お客様` |
| `FACTS` | 数値データの説明 | `TOTAL_ASSETS` は `総資産（円）` |
| `DIMENSIONS` | カテゴリデータの説明 | `SEGMENT` は `顧客セグメント` |
| `METRICS` | 集計指標の定義 | `SUM(TOTAL_ASSETS)` は `AUM（預かり資産合計）` |

In [ ]:
display_image('semantic-view.png')

In [ ]:
display_image('semantic-view-structure.png')

### 2-1. 顧客資産管理 Semantic View の作成

以下の6つのテーブル/ビューを統合した Semantic View を作成します：

- `DIM_CUSTOMER`: 顧客マスタ（主軸）
- `DIM_FAMILY`: 家族構成（相続人情報）
- `DIM_LIFE_EVENT`: ライフイベント（教育・相続等）
- `FACT_PORTFOLIO`: ポートフォリオ（保有資産）
- `FACT_TRANSACTION`: 取引履歴
- `V_INHERITANCE_TAX_ESTIMATE`: 相続税試算ビュー

> ⏱️ **目安: 10分**

In [ ]:
-- =============================================
-- Semantic View 1: 顧客資産管理
-- =============================================
CREATE OR REPLACE SEMANTIC VIEW SNOWFINANCE_DB.DEMO_SCHEMA.CUSTOMER_WEALTH_SEMANTIC_VIEW
TABLES (
    CUSTOMERS AS DIM_CUSTOMER
        PRIMARY KEY (CUSTOMER_ID)
        WITH SYNONYMS = ('顧客', 'お客様', 'クライアント', 'RM顧客')
        COMMENT = '顧客マスタ。富裕層50名の属性・担当RM情報',

    FAMILY AS DIM_FAMILY
        PRIMARY KEY (FAMILY_ID)
        WITH SYNONYMS = ('家族', '家族構成', '相続人')
        COMMENT = '顧客の家族構成。相続対策の提案に使用',

    LIFE_EVENTS AS DIM_LIFE_EVENT
        PRIMARY KEY (EVENT_ID)
        WITH SYNONYMS = ('ライフイベント', '生活イベント', '将来の予定')
        COMMENT = 'ライフイベント（教育・相続・不動産購入等）',

    PORTFOLIO AS FACT_PORTFOLIO
        PRIMARY KEY (PORTFOLIO_ID)
        WITH SYNONYMS = ('ポートフォリオ', '保有資産', '持ち株')
        COMMENT = '保有資産明細（含み益・含み損含む）',

    TRANSACTIONS AS FACT_TRANSACTION
        PRIMARY KEY (TRANSACTION_ID)
        WITH SYNONYMS = ('取引', '売買履歴', 'トランザクション')
        COMMENT = '取引履歴（売買・入出金）',

    INHERITANCE_TAX AS V_INHERITANCE_TAX_ESTIMATE
        PRIMARY KEY (CUSTOMER_ID)
        WITH SYNONYMS = ('相続税', '相続税試算', '推定相続税')
        COMMENT = '推定相続税試算ビュー'
)
RELATIONSHIPS (
    CUSTOMERS_TO_FAMILY AS CUSTOMERS(CUSTOMER_ID) REFERENCES FAMILY(CUSTOMER_ID),
    CUSTOMERS_TO_EVENTS AS CUSTOMERS(CUSTOMER_ID) REFERENCES LIFE_EVENTS(CUSTOMER_ID),
    CUSTOMERS_TO_PORTFOLIO AS CUSTOMERS(CUSTOMER_ID) REFERENCES PORTFOLIO(CUSTOMER_ID),
    CUSTOMERS_TO_TRANSACTIONS AS CUSTOMERS(CUSTOMER_ID) REFERENCES TRANSACTIONS(CUSTOMER_ID),
    CUSTOMERS_TO_INHERITANCE AS CUSTOMERS(CUSTOMER_ID) REFERENCES INHERITANCE_TAX(CUSTOMER_ID)
)
FACTS (
    CUSTOMERS.TOTAL_ASSETS AS TOTAL_ASSETS
        WITH SYNONYMS = ('総資産', '資産', '保有総額', '残高')
        COMMENT = '総資産（円）',
    CUSTOMERS.LIQUID_ASSETS AS LIQUID_ASSETS
        WITH SYNONYMS = ('流動性資産', '現金', '現預金')
        COMMENT = '流動性資産（円）',
    PORTFOLIO.MARKET_VALUE AS MARKET_VALUE
        WITH SYNONYMS = ('時価', '時価評価額', 'マーケットバリュー')
        COMMENT = '保有資産の時価評価額（円）',
    PORTFOLIO.UNREALIZED_GAIN AS UNREALIZED_GAIN
        WITH SYNONYMS = ('含み益', '含み損益', '未実現損益')
        COMMENT = '含み損益（円）。プラスが含み益、マイナスが含み損',
    INHERITANCE_TAX.ESTIMATED_TAX_AMOUNT AS ESTIMATED_TAX
        WITH SYNONYMS = ('相続税', '推定相続税', '相続税額')
        COMMENT = '推定相続税額（円）'
)
DIMENSIONS (
    CUSTOMERS.CUSTOMER_NAME AS CUSTOMER_NAME
        WITH SYNONYMS = ('顧客名', '氏名', '名前', 'お名前')
        COMMENT = '顧客氏名',
    CUSTOMERS.AGE AS AGE
        WITH SYNONYMS = ('年齢', '歳')
        COMMENT = '顧客の年齢（歳）',
    CUSTOMERS.SEGMENT AS SEGMENT
        WITH SYNONYMS = ('セグメント', '顧客ランク', 'ランク')
        COMMENT = '顧客セグメント（プラチナ/ゴールド/シルバー等）',
    CUSTOMERS.RISK_TOLERANCE AS RISK_TOLERANCE
        WITH SYNONYMS = ('リスク許容度', 'リスク', '運用スタイル')
        COMMENT = 'リスク許容度（積極運用/バランス/安定運用）',
    CUSTOMERS.INVESTMENT_PURPOSE AS INVESTMENT_PURPOSE
        WITH SYNONYMS = ('投資目的', '運用目的')
        COMMENT = '投資目的（資産形成/資産保全/相続対策等）',
    CUSTOMERS.RM_NAME AS RM_NAME
        WITH SYNONYMS = ('担当RM', '担当者', 'RM')
        COMMENT = '担当リレーションシップマネージャー名',
    PORTFOLIO.SECURITY_NAME AS SECURITY_NAME
        WITH SYNONYMS = ('銘柄', '株', '有価証券')
        COMMENT = '銘柄名',
    PORTFOLIO.ASSET_CLASS AS ASSET_CLASS
        WITH SYNONYMS = ('資産クラス', '投資種別')
        COMMENT = '資産クラス（株式/債券/投資信託等）',
    LIFE_EVENTS.EVENT_TYPE AS EVENT_TYPE
        WITH SYNONYMS = ('イベント種別', 'イベントタイプ')
        COMMENT = 'ライフイベント種別（教育資金/相続/不動産等）',
    LIFE_EVENTS.URGENCY AS URGENCY
        WITH SYNONYMS = ('緊急度', '優先度')
        COMMENT = 'ライフイベントの緊急度（高/中/低）'
)
METRICS (
    CUSTOMERS.SUM_TOTAL_ASSETS AS SUM(CUSTOMERS.TOTAL_ASSETS)
        WITH SYNONYMS = ('AUM', '預かり資産合計', '総預かり資産')
        COMMENT = '全顧客の総資産合計（円）',
    CUSTOMERS.AVG_TOTAL_ASSETS AS AVG(CUSTOMERS.TOTAL_ASSETS)
        WITH SYNONYMS = ('平均資産', '平均預かり資産')
        COMMENT = '顧客1人あたりの平均総資産（円）',
    CUSTOMERS.COUNT_CUSTOMERS AS COUNT(CUSTOMERS.CUSTOMER_ID)
        WITH SYNONYMS = ('顧客数', '顧客件数', '人数')
        COMMENT = '顧客数',
    PORTFOLIO.SUM_MARKET_VALUE AS SUM(PORTFOLIO.MARKET_VALUE)
        WITH SYNONYMS = ('ポートフォリオ評価額', '時価総額')
        COMMENT = '保有資産の時価評価総額',
    PORTFOLIO.SUM_UNREALIZED_GAIN AS SUM(PORTFOLIO.UNREALIZED_GAIN)
        WITH SYNONYMS = ('含み益合計', '含み損益合計')
        COMMENT = '含み損益合計（円）'
);

SELECT 'CUSTOMER_WEALTH_SEMANTIC_VIEW created!' AS STATUS;

### 2-2. 信託銀行商品 Semantic View の作成

信託商品と推奨ロジックのテーブルを統合した Semantic View を作成します。

In [ ]:
-- =============================================
-- Semantic View 2: 信託銀行商品
-- =============================================
CREATE OR REPLACE SEMANTIC VIEW SNOWFINANCE_DB.DEMO_SCHEMA.TRUST_PRODUCT_SEMANTIC_VIEW
    COMMENT = '信託銀行商品（証券担保ローン・教育信託等）の商品情報と推奨条件を管理するセマンティックビュー。対象セグメント・ライフイベントに応じた商品推奨ロジックを含む'
TABLES (
    PRODUCTS AS DIM_TRUST_PRODUCT
        PRIMARY KEY (PRODUCT_ID)
        WITH SYNONYMS = ('信託商品', '商品', 'プロダクト', '金融商品')
        COMMENT = '信託銀行商品マスタ（証券担保ローン・教育信託等）',
    RECOMMENDATIONS AS DIM_PRODUCT_RECOMMENDATION
        PRIMARY KEY (RECOMMENDATION_ID)
        WITH SYNONYMS = ('推奨条件', '適合条件', 'レコメンデーション')
        COMMENT = '商品推奨ロジック（対象セグメント・ライフイベント等）'
)
RELATIONSHIPS (
    RECOMMENDATIONS_TO_PRODUCTS AS RECOMMENDATIONS(PRODUCT_ID) REFERENCES PRODUCTS(PRODUCT_ID)
)
FACTS (
    PRODUCTS.MIN_AMOUNT AS MIN_AMOUNT
        WITH SYNONYMS = ('最小金額', '最低金額')
        COMMENT = '最小融資・投資金額（円）',
    PRODUCTS.MAX_AMOUNT AS MAX_AMOUNT
        WITH SYNONYMS = ('最大金額', '最高金額')
        COMMENT = '最大融資・投資金額（円）',
    PRODUCTS.INTEREST_RATE_MIN AS INTEREST_RATE_MIN
        WITH SYNONYMS = ('最低金利', '金利下限')
        COMMENT = '最低金利（%）',
    PRODUCTS.INTEREST_RATE_MAX AS INTEREST_RATE_MAX
        WITH SYNONYMS = ('最高金利', '金利上限')
        COMMENT = '最高金利（%）',
    RECOMMENDATIONS.TARGET_ASSETS_MIN AS TARGET_ASSETS_MIN
        WITH SYNONYMS = ('対象最小資産', '推奨対象資産下限')
        COMMENT = '推奨対象の最小資産規模（円）'
)
DIMENSIONS (
    PRODUCTS.PRODUCT_NAME AS PRODUCT_NAME
        WITH SYNONYMS = ('商品名', 'プロダクト名')
        COMMENT = '商品名',
    PRODUCTS.PRODUCT_CATEGORY AS PRODUCT_CATEGORY
        WITH SYNONYMS = ('商品カテゴリ', '商品種別')
        COMMENT = '商品カテゴリ（ローン/信託/その他）',
    PRODUCTS.TAX_BENEFIT AS TAX_BENEFIT
        WITH SYNONYMS = ('税制メリット', '節税効果', '税務上のメリット')
        COMMENT = '税制メリット（非課税枠・控除等）',
    RECOMMENDATIONS.TARGET_LIFE_EVENT AS TARGET_LIFE_EVENT
        WITH SYNONYMS = ('対象ライフイベント', '推奨イベント')
        COMMENT = '推奨対象のライフイベント種別',
    RECOMMENDATIONS.RECOMMENDATION_REASON AS RECOMMENDATION_REASON
        WITH SYNONYMS = ('推奨理由', '提案理由')
        COMMENT = 'この商品を推奨する理由'
)
METRICS (
    PRODUCTS.COUNT_PRODUCTS AS COUNT(PRODUCTS.PRODUCT_ID)
        WITH SYNONYMS = ('商品数', '取扱商品数')
        COMMENT = '商品数'
);

SELECT 'TRUST_PRODUCT_SEMANTIC_VIEW created!' AS STATUS;

In [ ]:
%%sql -r result_show_sv
-- 作成された Semantic View の一覧確認
SHOW SEMANTIC VIEWS IN SCHEMA DEMO_SCHEMA;

In [ ]:
%%sql -r result_desc_sv
-- Semantic View の構成を確認（顧客資産管理）
DESCRIBE SEMANTIC VIEW SNOWFINANCE_DB.DEMO_SCHEMA.CUSTOMER_WEALTH_SEMANTIC_VIEW;

## 3. Cortex Analyst のテスト（REST API経由）

Snowflake Notebook から Cortex Analyst の REST API を直接呼び出してテストできます。

In [ ]:
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()

def ask_cortex_analyst(question: str, semantic_view: str = "SNOWFINANCE_DB.DEMO_SCHEMA.CUSTOMER_WEALTH_SEMANTIC_VIEW") -> dict:
    """Cortex Analyst REST API を呼び出す"""
    request_body = {
        "messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
        "semantic_view": semantic_view
    }
    resp = session.connection.rest.request(
        url="/api/v2/cortex/analyst/message",
        body=request_body,
        method="post",
        client="rest",
        timeout=60,
    )
    return resp

def ask_and_run(question: str, semantic_view: str = "SNOWFINANCE_DB.DEMO_SCHEMA.CUSTOMER_WEALTH_SEMANTIC_VIEW"):
    """Cortex Analyst に質問し、生成されたSQLを実行して結果を表示"""
    print(f"質問: {question}")
    print("=" * 60)
    result = ask_cortex_analyst(question, semantic_view)
    generated_sql = None
    for msg in result.get("message", {}).get("content", []):
        if msg["type"] == "text":
            print(msg["text"])
        elif msg["type"] == "sql":
            generated_sql = msg["statement"]
            print(f"\n【生成されたSQL】\n{generated_sql}")
    if generated_sql:
        print("\n【クエリ結果】")
        df = session.sql(generated_sql).to_pandas()
        display(df)
    return result

# テスト: 顧客資産分析
result = ask_and_run("預かり資産5億円以上の顧客を資産が多い順にリストアップしてください")

## 4. Snowsight の Cortex Analyst UI でのテスト

### GUI でのアクセス手順

1. **Snowsight** にログイン
2. 左メニュー「**AI と ML**」→「**Cortex Analyst**」をクリック
3. 「**+ New**」ボタンをクリック
4. Semantic View として `SNOWFINANCE_DB.DEMO_SCHEMA.CUSTOMER_WEALTH_SEMANTIC_VIEW` を選択
5. 下部のチャットボックスに質問を入力

### AI Studio からのアクセス方法（別ルート）

1. 左メニュー「**AI Studio**」をクリック
2. 「**Cortex Analyst**」タブを選択
3. Semantic View を選択して質問

> 💡 **ポイント**: Semantic View に `SYNONYMS` を設定しているため、「顧客」「お客様」「クライアント」などどれで質問しても同じ結果が返ります！

### テスト質問例（自然言語）

以下の質問を Cortex Analyst UI に入力してみましょう。
また、下のセルでは Cortex Analyst が生成するSQLに近いクエリを手動で実行して、結果を確認できます。

---

#### CUSTOMER_WEALTH_SEMANTIC_VIEW で試す質問

**Q1**: 「預かり資産が5億円以上の顧客をセグメント別に集計してください」

**Q2**: 「教育資金のライフイベントがあり、緊急度が高い顧客をリストアップしてください」

**Q3**: 「担当RM別の預かり資産合計と顧客数を教えてください」

**Q4**: 「トヨタ株を1億円以上保有している顧客は何人いますか？」

**Q5**: 「相続税の試算が1億円を超える顧客のリストと、それぞれの総資産を教えてください」

---

#### TRUST_PRODUCT_SEMANTIC_VIEW で試す質問

**Q6**: 「証券担保ローンの最低金利と最高金利を教えてください」

**Q7**: 「教育資金に関連するライフイベントに対応した商品と、その推奨理由を教えてください」

> ⏱️ **目安: 15分**（各質問を入力して結果を確認）

In [ ]:
%%sql -r result_q1
-- Q1: 資産5億円以上の顧客をセグメント別に集計
SELECT
    SEGMENT AS "セグメント",
    COUNT(*) AS "顧客数",
    TO_CHAR(SUM(TOTAL_ASSETS)/100000000, '9,999') AS "預かり資産合計億円",
    TO_CHAR(AVG(TOTAL_ASSETS)/100000000, '9,999.0') AS "平均資産億円"
FROM DIM_CUSTOMER
WHERE TOTAL_ASSETS >= 500000000
GROUP BY SEGMENT
ORDER BY SUM(TOTAL_ASSETS) DESC;

In [ ]:
-- Q2: 教育資金のライフイベントがあり緊急度が高い顧客
SELECT
    c.CUSTOMER_ID,
    c.CUSTOMER_NAME,
    c.AGE AS "年齢",
    e.EVENT_TYPE AS "イベント種別",
    e.EXPECTED_DATE AS "予定日",
    e.EVENT_DETAIL AS "内容",
    c.RM_NAME AS "担当RM",
    TO_CHAR(c.TOTAL_ASSETS/100000000, '9,999.0') AS "総資産億円"
FROM DIM_CUSTOMER c
JOIN DIM_LIFE_EVENT e ON c.CUSTOMER_ID = e.CUSTOMER_ID
WHERE e.EVENT_TYPE LIKE '%教育%'
  AND e.URGENCY = '高'
ORDER BY e.EXPECTED_DATE ASC;

In [ ]:
%%sql -r result_q3
-- Q3: 担当RM別の預かり資産合計と顧客数
SELECT
    RM_NAME AS "担当RM",
    COUNT(*) AS "顧客数",
    TO_CHAR(SUM(TOTAL_ASSETS)/100000000, '9,999') AS "預かり資産合計億円",
    TO_CHAR(AVG(TOTAL_ASSETS)/100000000, '9,999.0') AS "平均資産億円",
    TO_CHAR(MAX(TOTAL_ASSETS)/100000000, '9,999.0') AS "最大顧客資産億円"
FROM DIM_CUSTOMER
GROUP BY RM_NAME
ORDER BY SUM(TOTAL_ASSETS) DESC;

In [ ]:
%%sql -r result_q4
-- Q4: トヨタ株（7203）を1億円以上保有している顧客
SELECT
    c.CUSTOMER_ID,
    c.CUSTOMER_NAME,
    c.AGE AS "年齢",
    p.SECURITY_NAME AS "銘柄",
    TO_CHAR(p.MARKET_VALUE, '999,999,999') AS "時価評価額円",
    ROUND(p.UNREALIZED_GAIN_PCT, 1) AS "含み損益率",
    c.RM_NAME AS "担当RM"
FROM DIM_CUSTOMER c
JOIN FACT_PORTFOLIO p ON c.CUSTOMER_ID = p.CUSTOMER_ID
WHERE p.SECURITY_CODE = '7203'
  AND p.MARKET_VALUE >= 100000000
ORDER BY p.MARKET_VALUE DESC;

In [ ]:
-- Q5: 相続税試算が1億円を超える顧客（上位10名）
SELECT
    v.CUSTOMER_ID,
    v.CUSTOMER_NAME,
    v.AGE AS "年齢",
    TO_CHAR(v.TOTAL_ASSETS/100000000, '9,999.0') AS "総資産億円",
    TO_CHAR(v.ESTIMATED_TAX/100000000, '9,999.0') AS "推定相続税億円",
    ROUND(v.ESTIMATED_TAX / v.TOTAL_ASSETS * 100, 1) AS "相続税率",
    c.RM_NAME AS "担当RM"
FROM V_INHERITANCE_TAX_ESTIMATE v
JOIN DIM_CUSTOMER c ON v.CUSTOMER_ID = c.CUSTOMER_ID
WHERE v.ESTIMATED_TAX >= 100000000
ORDER BY v.ESTIMATED_TAX DESC
LIMIT 10;

In [ ]:
%%sql -r result_q6
-- Q6: 山田様（C001）の属性に合う信託商品のマッチング
-- 年齢・資産・ライフイベントから最適商品を推奨
SELECT
    p.PRODUCT_NAME AS "商品名",
    p.PRODUCT_CATEGORY AS "カテゴリ",
    p.DESCRIPTION AS "概要",
    TO_CHAR(p.MIN_AMOUNT/100000000, '9,999.0') || '億円〜' AS "最小金額",
    p.TAX_BENEFIT AS "税制メリット",
    r.RECOMMENDATION_REASON AS "推奨理由"
FROM DIM_TRUST_PRODUCT p
JOIN DIM_PRODUCT_RECOMMENDATION r ON p.PRODUCT_ID = r.PRODUCT_ID
JOIN DIM_CUSTOMER c ON c.CUSTOMER_ID = 'C001'
WHERE (r.TARGET_AGE_MIN IS NULL OR c.AGE >= r.TARGET_AGE_MIN)
  AND (r.TARGET_AGE_MAX IS NULL OR c.AGE <= r.TARGET_AGE_MAX)
  AND (r.TARGET_ASSETS_MIN IS NULL OR c.TOTAL_ASSETS >= r.TARGET_ASSETS_MIN)
  AND p.IS_ACTIVE = TRUE
ORDER BY r.PRIORITY ASC;

## まとめ

Part 2 完了！Cortex Analyst のセットアップが完了しました。

### 作成したオブジェクト

| Semantic View | 対象 | テーブル数 | 主な指標 |
|---|---|---|---|
| `CUSTOMER_WEALTH_SEMANTIC_VIEW` | 顧客資産管理 | 6テーブル | AUM, 相続税試算, 含み益 |
| `TRUST_PRODUCT_SEMANTIC_VIEW` | 信託銀行商品 | 2テーブル | 商品条件, 推奨理由 |

### Semantic View の威力

- **同義語対応**: 「顧客」「お客様」「クライアント」どれで質問しても同じ結果
- **複数テーブル結合**: 「顧客の相続税と保有銘柄を合わせて教えて」も自動的にJOIN
- **ビジネス用語変換**: `TOTAL_ASSETS` を「総資産・AUM・預かり資産」として認識
- **日本語対応**: 日本語でも高精度に自然言語 to SQL が動作

### よくある質問

**Q: Semantic View と通常のVIEWの違いは？**
A: 通常のVIEWはSQLで定義されたデータ変換ですが、Semantic ViewはAIへのメタデータ定義です。SQLは含まず、テーブルの「意味」のみを定義します。

**Q: Cortex Analyst はどのモデルを使っていますか？**
A: Snowflake が管理する専用モデルを使用しています。ユーザーがモデルを選択する必要はありません。

**次のステップ:** `part4_cortex_search.ipynb` で Cortex Search（セマンティック検索）を体験してください。